In [4]:
import pandas as pd 
import numpy as np 
df1=pd.read_csv('/content/tmdb_5000_credits.csv')
df2=pd.read_csv('/content/tmdb_5000_movies.csv')
ratings = pd.read_csv('/ratings_small.csv')

# Importamos linear_kernel
from sklearn.metrics.pairwise import linear_kernel
#Importamos TfIdfVectorizer de scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from datetime import  datetime

Para crear este Jupyter Notebook, nos basamos fuertemente en la documentación del paquete Surprise y de Movie Recommendation System, que se encuentran en las referencias.

Un tipo de recomendación es la recomendación demográfica, en el cual las recomendaciones se dan a partir de la popularidad de la película en cuestión. Este sistema no toma en cuenta otro factor. Es una recomendación general de películas de acuerdo a la popularidad. La idea general es que las películas más populares tienen más probabilidad de gustar al público en general.


In [ ]:
df1.columns = ['id','tittle','cast','crew']
df2= df2.merge(df1,on='id')

In [ ]:
df2.head(5)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,tittle,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


Para hacer el filtro de películas más populares, no solo basta conocer el score, ya que este puede ser alto, sin embargo, podría ser que fuera votado por algunas personas. Podemos tomar en cuenta una métrica utilizada por IMbd: promedio ponderado de los ratings. 

v  es el número de votos que tiene la película;

m  es el mínimo de votos requeridos para poder aparecer en nuestra recomendación;

R  es la calificación promedio de la película;

C es el promedio general de todas las películas


(v/m+v * R) + (m/m+v * C)

Para obtener C, solamente sacamos el promedio de todas las películas

In [ ]:
C= df2['vote_average'].mean()
C

6.092171559442011

Para obtener el mínimo de votos requeridos, podemos tomar el cuantil 85 de películas  con vote count más alto

In [ ]:
m= df2['vote_count'].quantile(0.85)
m

1300.6999999999998

Solo tomaremos en cuenta películas con un mínimo de votos m y nos queda una dataframe de 721 películas.

In [ ]:
q_movies = df2.copy().loc[df2['vote_count'] >= m]
q_movies.shape

(721, 23)

Realizamos el promedio ponderado como lo definimos más arriba y la aplicamos a nuestro DF con películas de cuantil 85:

In [ ]:
def promedio_ponderado(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v/(v+m) * R) + (m/(m+v) * C)

In [ ]:
q_movies['score'] = q_movies.apply(promedio_ponderado, axis=1)

In [ ]:
#Reacomodamos de forma descendente
q_movies = q_movies.sort_values('score', ascending=False)

#Se imprimen las 15 recomendaciones de películas (que son las que tienen un score más alto)
q_movies[['title', 'vote_count', 'vote_average', 'score']].head(10)

,title,vote_count,vote_average,score
1881,The Shawshank Redemption,8205,8.5,8.170528
662,Fight Club,9413,8.3,8.031958
3232,Pulp Fiction,8428,8.3,8.004820
65,The Dark Knight,12002,8.2,7.993903
3337,The Godfather,5893,8.4,7.982719
96,Inception,13752,8.1,7.926504
809,Forrest Gump,7927,8.2,7.902889
95,Interstellar,10867,8.1,7.885368
329,The Lord of the Rings: The Return of the King,8064,8.1,7.821125
1990,The Empire Strikes Back,5879,8.2,7.818138


Esta es una forma muy básica de hacer recomendaciones, sin embargo, podemos rescatar el promedio ponderado para evitar el sesgo de películas poco votadas.

Otra forma de hacer recomendaciones es el filtrado basado en contenido. En este sistema de recomendación se utiliza el contenido de la película (descripción general, reparto, equipo, palabra clave, eslogan, etc.) para encontrar su similitud con otras películas.

En la variable overview, tenemos la descripción general de la película:

In [ ]:
df2['overview'].head(5)

0    In the 22nd century, a paraplegic Marine is di...
1    Captain Barbossa, long believed to be dead, ha...
2    A cryptic message from Bond’s past sends him o...
3    Following the death of District Attorney Harve...
4    John Carter is a war-weary, former military ca...
Name: overview, dtype: object

Vamos a convertir el vector de palabras de cada entrada de overview en un Term Frequency-Inverse Document Frequency (TF-IDF).

La frecuencia es la frecuencia relativa de una palabra en un documento y se da como instancia de: términos/ instancias totales.

La frecuencia inversa de frecuencia de documentos es el recuento relativo de documentos que contiene un registros: log(número de documentos / documentos con término).

LA importancia general de cada palabra para los documentos en los que aparece es igual a la multiplicación de frecuencia de términos por la inversa de frecuencia de documentos(Term Frequency y Inverse Document Frequency): 
TF * IDF

Esto nos dará una matriz donde cada columna representa una palabra en el vocabulario general (todas las palabras que aparecen en al menos un documento) y cada fila representa una película, como antes. Esto se hace para reducir la importancia de las palabras que ocurren con frecuencia. en las descripciones generales de la trama y, por lo tanto, su importancia en el cálculo de la puntuación final de similitud.


In [ ]:
#Importamos TfIdfVectorizer de scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer

#Definimos al TF-IDF Vectorizer Object. Remove all english stop words such as 'the', 'a'
tfidf = TfidfVectorizer(stop_words='english')

#Quitamos Nan con espacios vacíos
df2['overview'] = df2['overview'].fillna('')

#Construimos la matriz TF-IDF:
tfidf_matrix = tfidf.fit_transform(df2['overview'])

#OLa matriz tfidf_matrix tiene la siguiente forma:
tfidf_matrix.shape
print(tfidf_matrix)

  (0, 3545)	0.27493285363270825
  (0, 795)	0.2084074203878777
  (0, 14754)	0.26556705630591243
  (0, 13355)	0.2537072195025191
  (0, 7414)	0.2153609580612842
  (0, 19054)	0.23864791282019557
  (0, 12269)	0.1778447209530258
  (0, 19671)	0.24152597853522884
  (0, 13644)	0.29248619979162926
  (0, 12411)	0.27155366980600426
  (0, 5519)	0.27863433071832444
  (0, 11667)	0.2580267910376776
  (0, 13680)	0.3411386462681145
  (0, 3232)	0.19886014555883547
  (0, 225)	0.3055534742040413
  (1, 14969)	0.2764509091386784
  (1, 18376)	0.37466189442120984
  (1, 6188)	0.2743698871995736
  (1, 19418)	0.3060153616124342
  (1, 6016)	0.1966138874334538
  (1, 6069)	0.2948248727159851
  (1, 8674)	0.2809447973706985
  (1, 11025)	0.12158034118460531
  (1, 3843)	0.19195626583733216
  (1, 4854)	0.19967394354414503
  :	:
  (4802, 5841)	0.34247058916908363
  (4802, 4576)	0.15002600185636672
  (4802, 10126)	0.15121328276821538
  (4802, 16409)	0.17123529458454181
  (4802, 4816)	0.14223771161926407
  (4802, 1311)	0.16

Podemos ver que 20,978 palabras fueron utilizadas para explicar 4803 películas de nuestro data set.


Ahora podremos calcular la similitud de películas en nuestro dataset.

Podemos utilizar la similitud del coseno para calcular una cantidad numérica que denota la similitud entre dos películas. Usamos la puntuación de similitud de coseno. Y dado que usamos el vectorizador TF-IDF, el cálculo del producto punto, nos dará directamente la puntuación de similitud del coseno.

In [ ]:
# Import linear_kernel
from sklearn.metrics.pairwise import linear_kernel

# Construimos la matriz de similitud de coseno
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
print(cosine_sim)

[[1.         0.         0.         ... 0.         0.         0.        ]
 [0.         1.         0.         ... 0.02160533 0.         0.        ]
 [0.         0.         1.         ... 0.01488159 0.         0.        ]
 ...
 [0.         0.02160533 0.01488159 ... 1.         0.01609091 0.00701914]
 [0.         0.         0.         ... 0.01609091 1.         0.01171696]
 [0.         0.         0.         ... 0.00701914 0.01171696 1.        ]]


Para realizar una función que tome como input el título de la película y nos arroje una lista de las 10 películas más parecidas, tenemos que identificar a cada película con un índice.

In [ ]:
#Construimos un dataframe de título de película con su índice
indices = pd.Series(df2.index, index=df2['title']).drop_duplicates()
print(indices)

title
Avatar                                         0
Pirates of the Caribbean: At World's End       1
Spectre                                        2
The Dark Knight Rises                          3
John Carter                                    4
                                            ... 
El Mariachi                                 4798
Newlyweds                                   4799
Signed, Sealed, Delivered                   4800
Shanghai Calling                            4801
My Date with Drew                           4802
Length: 4803, dtype: int64


Función de recomendación:

Obtenga la lista de puntuaciones de similitud de coseno para esa película en particular con todas las películas. 

Conviértalo en una lista de tuplas donde el primer elemento es su posición y el segundo es la puntuación de similitud.

Ordene la lista de tuplas antes mencionada según las puntuaciones de similitud; es decir, el segundo elemento.

Obtenga los 10 elementos principales de esta lista. Ignore el primer elemento, ya que se refiere a sí mismo (la película más similar a una película en particular es la película en sí).


Devuelve los títulos correspondientes a los índices de los elementos superiores.

In [ ]:
# Function de recomendación
def get_recommendations(title, cosine_sim=cosine_sim):
    # Obtenemos el índicde de la película
    idx = indices[title]

    # Obtenemos las puntuaciones de similitud por pares de todas las películas con esa películ
    #Se hace por tupla, el primer elemento corresponde al índice de la película, el segundo a su similitud
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Ordenamos según su similitud
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Obtenemos los 10 más similares
    sim_scores = sim_scores[1:11]

    # Obtenemos los índices
    movie_indices = [i[0] for i in sim_scores]

    # Regresamos las películas más similares
    return df2['title'].iloc[movie_indices]

In [ ]:
get_recommendations('White Chicks')

2511                        Home Alone
186                        Bad Boys II
4529                 Hurricane Streets
753                       The Sentinel
2269              Butterfly on a Wheel
2086    Home Alone 2: Lost in New York
4726                        The Mighty
1422    The X Files: I Want to Believe
731                         The Jackal
2813                  Think Like a Man
Name: title, dtype: object

El algoritmo es bueno, sin embargo, no refleja o no captura otras características, solo la similitud de las películas. Tenemos como supuesto que la gente le gustarán películas con contenido similar, lo cual no siempre aplica. 


Podemos ver el problema de recomendación como uno de  optimización. Podemos verlo como lo buenos que somos al predecir la calificación que un usuario dará a una película, dados sus ratings pasados.

Utilizaremos el factor latente


In [3]:
!pip install surprise
from surprise import SVD
from surprise import Dataset
from surprise.model_selection import cross_validate
from surprise import Reader, Dataset, SVD

     |████████████████████████████████| 11.8 MB 50 kB/s 
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.1-cp37-cp37m-linux_x86_64.whl size=1619432 sha256=1fb8ce4bf0a5722edcf21966ada7a4abc6ebf7d80cdd3ebd6af524ec62520431
  Stored in directory: /root/.cache/pip/wheels/76/44/74/b498c42be47b2406bd27994e16c5188e337c657025ab400c1c
Successfully built scikit-surprise


In [5]:
reader = Reader()
ratings.head()

ratings['Time'] = ratings['timestamp'].apply(datetime.fromtimestamp)
print(ratings)

        userId  movieId  rating   timestamp                Time
0            1       31     2.5  1260759144 2009-12-14 02:52:24
1            1     1029     3.0  1260759179 2009-12-14 02:52:59
2            1     1061     3.0  1260759182 2009-12-14 02:53:02
3            1     1129     2.0  1260759185 2009-12-14 02:53:05
4            1     1172     4.0  1260759205 2009-12-14 02:53:25
...        ...      ...     ...         ...                 ...
99999      671     6268     2.5  1065579370 2003-10-08 02:16:10
100000     671     6269     4.0  1065149201 2003-10-03 02:46:41
100001     671     6365     4.0  1070940363 2003-12-09 03:26:03
100002     671     6385     2.5  1070979663 2003-12-09 14:21:03
100003     671     6565     3.5  1074784724 2004-01-22 15:18:44

[100004 rows x 5 columns]


In [6]:
#Leemos la base de datos para poder ser utilizada por la librería
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# Utilizamos el algoritmo de SVD: Singular Value Decomposition, que sirve para hacer modelos predictivos. 
algorithm_1 = SVD()

# Hacemos validación cruzada 5 para medir el error de testing
cross_validate(algorithm_1, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8913  0.9007  0.8958  0.9003  0.9015  0.8979  0.0038  
MAE (testset)     0.6877  0.6958  0.6891  0.6952  0.6937  0.6923  0.0033  
Fit time          5.39    5.39    5.44    5.39    5.22    5.37    0.07    
Test time         0.18    0.22    0.29    0.17    0.19    0.21    0.04    


{'fit_time': (5.388938665390015,
  5.393181562423706,
  5.4416162967681885,
  5.394934415817261,
  5.224609613418579),
 'test_mae': array([0.68765024, 0.69576657, 0.68908419, 0.69522954, 0.69373398]),
 'test_rmse': array([0.89134686, 0.90072989, 0.89579872, 0.90028665, 0.90148873]),
 'test_time': (0.17507553100585938,
  0.21695613861083984,
  0.29248714447021484,
  0.17420125007629395,
  0.19454216957092285)}

In [7]:
#Ahora podemos tomar un userID como referencia
ratings[ratings['userId'] == 1]

,userId,movieId,rating,timestamp,Time
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02
3,1,1129,2.0,1260759185,2009-12-14 02:53:05
4,1,1172,4.0,1260759205,2009-12-14 02:53:25
5,1,1263,2.0,1260759151,2009-12-14 02:52:31
6,1,1287,2.0,1260759187,2009-12-14 02:53:07
7,1,1293,2.0,1260759148,2009-12-14 02:52:28
8,1,1339,3.5,1260759125,2009-12-14 02:52:05
9,1,1343,2.0,1260759131,2009-12-14 02:52:11


In [8]:
# Obtenemos una predicción de rating dependiendo de una película
#En este caso vamos a predecir del UserID 40, qué rating le daría a la película 302.
pred = algorithm_1.predict(uid=40, iid=302, r_ui=3)
print(pred)

user: 40         item: 302        r_ui = 3.00   est = 3.84   {'was_impossible': False}


Para este caso, el modelo predice que el cliente 40 o usuario 40 le otorgará una calificación de 3.84 a la película 302


El modelo es muy bueno y funciona bastante bien, en cuanto a código, la librería surprise es muy eficiente, sin embargo, si no tenemos información de os gustos de los clientes, no podríamos inferir información nueva. 
Podemos agregar más variables al modelo, es decir, predecir con base en contenido y otras variables importantes.

Fuentes:

[Movie Recommendation System](https://www.kaggle.com/ibtesama/getting-started-with-a-movie-recommendation-system/notebook)



[Surprise, Python](http://surpriselib.com/)


[Surprise, Python: getting started](https://surprise.readthedocs.io/en/stable/getting_started.html)